# 歷史版本 B：Update 05/23 – PCA

**用途**：05/23 版本的 PCA（相關矩陣、排除連江縣、PC1–PC4 負荷量與得分圖）。

| 項目 | 說明 |
|---|---|
| 輸入 | data/raw CSV |
| 輸出 | （歷史探索，圖表為即時輸出） |
| 來源 | MVA 期末報告分析.ipynb（Colab 原檔，191 cells） 之 cell 28–73 |

> 歷史版本，保留供追溯；不保證可獨立執行，最終流程請見 ../final_analysis/ 與 ../../src/。


In [ ]:
# 共用設定：載入資料並提供 df / pivot_table_age_std / pivot_table_cleaned / scaled_data / taiwan_map / prop 等變數
# （取代原 Colab 的 Drive 掛載、字型下載與 GDAL 設定；詳見 _legacy_setup.py）
import sys; sys.path.insert(0, '..')   # _legacy_setup.py 位於 notebooks/
from _legacy_setup import *
df = pd.read_csv(CSV_PATH)  # 早期版本使用原始中文欄位名稱（覆蓋共用設定的英文欄位版）

# Update 05/23

In [ ]:
df.columns = [
    'Diagnosis_Year', 'Gender', 'City_County', 'Cancer_Type',
    'Age_Standardized_Incidence_Rate', 'Cancer_Incidence_Count',
    'Mean_Age', 'Median_Age', 'Crude_Incidence_Rate'
]
# Display the new column names and a preview of the data
print("New column names:", df.columns.tolist())
display(df.head())

In [ ]:
unique_cancer_types = df['Cancer_Type'].unique()
print(unique_cancer_types)

$$
標準化死亡率= {\sum(標準人口各年齡層之加權數×觀察族群各年齡層死亡率)}
$$

<br>
$\text{標準人口} = $ WHO 2000 年世界標準人口

In [ ]:
df_filtered = df[(df['Gender'] == '全') &(df['City_County'] != '全國') & (df['Cancer_Type'] == '肺、支氣管及氣管')]
display(df_filtered)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='City_County',
    columns='Diagnosis_Year',
    values='Age_Standardized_Incidence_Rate',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)

In [ ]:
corr_matrix_yr = pivot_table_age_std.corr()
display(corr_matrix_yr.head())

In [ ]:
corr_matrix_county = pivot_table_age_std.T.corr()
display(corr_matrix_county.head())

In [ ]:
corr_matrix_county.columns

Now, let's use hierarchical clustering to reorder the correlation matrix for better visualization in the heatmap.

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import seaborn as sns

# Perform hierarchical clustering
linked = sch.linkage(corr_matrix_yr, method='ward')

# Get the order of the clusters
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix based on the clustering result
reordered_corr_matrix_yr = corr_matrix_yr.iloc[cluster_order, cluster_order]
display(reordered_corr_matrix_yr.head())

In [ ]:
# Perform hierarchical clustering
linked = sch.linkage(corr_matrix_county, method='ward')

# Get the order of the clusters
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix based on the clustering result
reordered_corr_matrix_county = corr_matrix_county.iloc[cluster_order, cluster_order]
display(reordered_corr_matrix_county.head())

Finally, let's visualize the reordered correlation matrix using a heatmap.

In [ ]:
# Re-draw the heatmap with fixed range
plt.figure(figsize=(12, 10))
sns.heatmap(reordered_corr_matrix_yr, annot=False, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Age-Standardized Incidence Rates by Years (Ordered by Hclust)')
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(reordered_corr_matrix_county, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Age-Standardized Incidence Rates by Counties (Ordered by Hclust)')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure Chinese font is applied
if 'prop' in locals() or 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

# Calculate base font sizes (increasing by 20%)
base_size = 12 * 1.1
title_size = 16 * 1.1
label_size = 14 * 1.1

plt.figure(figsize=(14, 8))
sns.lineplot(data=df_filtered, x='Diagnosis_Year', y='Age_Standardized_Incidence_Rate', hue='City_County')

plt.title('肺癌—年齡標準化發生率—按縣市與年度', fontsize=title_size)
plt.xlabel('年', fontsize=label_size)
plt.ylabel('年齡標準化發生率 (每10萬人口)', fontsize=label_size)
plt.legend(title='縣市別', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=base_size, title_fontsize=label_size)
plt.xticks(fontsize=base_size)
plt.yticks(fontsize=base_size)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the Chinese font is applied for the plot
if 'prop' in locals() or 'prop' in globals():
    plt.rcParams['font.family'] = prop.get_name()

plt.figure(figsize=(14, 8))
sns.lineplot(data=df_filtered, x='Diagnosis_Year', y='Median_Age', hue='City_County')

plt.title('Median Age of Cancer Patients by Year and City/County')
plt.xlabel('Year')
plt.ylabel('Median Age')
plt.legend(title='City/County', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.tight_layout()
plt.show()

## Principal Component Analysis (PCA) on Correlation Matrix (County)

Now, let's perform PCA on the `correlation_matrix` to reduce dimensionality and identify underlying patterns. We'll start by importing `PCA` from `sklearn.decomposition` and fitting it to our data.

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Initialize PCA. We will keep all components for the scree plot first.
pca = PCA()
pca.fit(corr_matrix_county)

# Explained variance ratio
explained_variance_ratio = pca.explained_variance_ratio_

# Cumulative explained variance
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print("Explained Variance Ratio:")
print(explained_variance_ratio)
print("\nCumulative Explained Variance:")
print(cumulative_explained_variance)

### Scree Plot

A scree plot helps visualize the eigenvalues (explained variance) for each principal component, allowing us to determine the optimal number of components to retain.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o', linestyle='--')
plt.title('Scree Plot')
plt.xlabel('Number of Components')
plt.ylabel('Explained Variance Ratio')
plt.grid(True)
plt.show()

### Cumulative Explained Variance Plot

This plot shows the cumulative sum of explained variance, which helps in deciding how many principal components are needed to explain a certain percentage of the total variance.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, marker='o', linestyle='--')
plt.axhline(y=0.8, color='r', linestyle='-', label='80% Cumulative Variance')
plt.title('Cumulative Explained Variance Plot')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.grid(True)
plt.legend()
plt.show()

### Biplot

A biplot visualizes the relationship between the observations (rows) and the variables (columns) in the PCA space. It helps in understanding how original variables contribute to the principal components and how observations are grouped.

In [ ]:
!pip install adjustText

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Re-calculating PCA components and explained variance ratio for corr_matrix_county
# This is needed for cell WRFIllJFiL6F
pca_2_components = PCA(n_components=2)
components = pca_2_components.fit_transform(corr_matrix_county)
explained_variance_ratio = pca_2_components.explained_variance_ratio_

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering

# 1. 取得縣市名稱列表
county_names = corr_matrix_county.index.tolist()

# 2. 利用你之前的階層分群設定，假設我們切成 4 群
hclust = AgglomerativeClustering(n_clusters=4, linkage='ward')
cluster_labels = hclust.fit_predict(corr_matrix_county)

# 3. 建立 PCA 畫圖用的 DataFrame
pca_df = pd.DataFrame({
    'PC1': components[:, 0],
    'PC2': components[:, 1],
    'County': county_names,
    'Cluster': [f'Cluster {c}' for c in cluster_labels]
})

# 4. 開始畫圖
plt.figure(figsize=(14, 10))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Cluster', palette='Set1', s=150, alpha=0.8)

# Determine the font properties to use for annotations
annotation_font_prop = None
if 'prop' in globals() and prop is not None: # Defaulting to Noto Sans TC if available
    annotation_font_prop = prop
elif 'prop_serif' in globals() and prop_serif is not None: # Fallback to Noto Serif TC
    annotation_font_prop = prop_serif

# 加註縣市名稱
for i, txt in enumerate(pca_df['County']):
    # Corrected argument: 'fontproperties' instead of 'prop'
    plt.annotate(txt, (pca_df['PC1'].iloc[i], pca_df['PC2'].iloc[i]),
                 textcoords="offset points", xytext=(5,5), ha='center', fontsize=11,
                 fontproperties=annotation_font_prop) # Use fontproperties here

plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
plt.title('PCA Biplot of Taiwan Counties based on Lung Cancer Trends', fontsize=16)
plt.xlabel(f'PC1 ({explained_variance_ratio[0]*100:.2f}%)', fontsize=12)
plt.ylabel(f'PC2 ({explained_variance_ratio[1]*100:.2f}%)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(title='Hclust Groups')
plt.show()

In [ ]:
loadings

In [ ]:
components

## 排除連江縣做 PCA

In [ ]:
# Check for rows (counties) with NaN values
rows_with_nan = pivot_table_age_std[pivot_table_age_std.isna().any(axis=1)].index.tolist()

# Check for columns (years) with NaN values
cols_with_nan = pivot_table_age_std.columns[pivot_table_age_std.isna().any(axis=0)].tolist()

print(f"Rows (Counties) with NaN values: {rows_with_nan}")
print(f"Columns (Years) with NaN values: {cols_with_nan}")

# Display the exact locations (Row, Column) where values are missing
if rows_with_nan or cols_with_nan:
    print("\nDetailed NaN locations:")
    nan_locations = pivot_table_age_std.isna().stack()
    nan_locations = nan_locations[nan_locations].index.tolist()
    for row, col in nan_locations:
        print(f"- County: {row}, Year: {col}")
else:
    print("No NaN values found in the table.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering

# ==========================================
# 1. 資料處理 (Drop NaNs) 與標準化
# ==========================================
# 排除含有缺失值的連江縣
pivot_table_cleaned = pivot_table_age_std.drop(index='連江縣')

scaler = StandardScaler()
scaled_data = scaler.fit_transform(pivot_table_cleaned)

# ==========================================
# 2. 執行 PCA
# ==========================================
pca_full = PCA()
pca_full.fit(scaled_data)

print("--- PCA 變異解釋量分析 (排除連江縣) ---")
print("各主成分解釋變異比 (前5個):", pca_full.explained_variance_ratio_[:5])
print("累積解釋變異比 (前5個):", np.cumsum(pca_full.explained_variance_ratio_)[:5])
print("-" * 30)

# 取前兩個主成分做後續分析
n_comps = 2
pca_2d = PCA(n_components=n_comps)
scores_matrix = pca_2d.fit_transform(scaled_data) # 縣市的新座標

# 計算 Loadings (年份與 PC 的相關性)
loadings_matrix = pca_2d.components_.T * np.sqrt(pca_2d.explained_variance_)

# ==========================================
# 3. 縣市分群 (Hclust)
# ==========================================
hclust = AgglomerativeClustering(n_clusters=4, linkage='ward')
cluster_labels = hclust.fit_predict(scaled_data)

# 建立 Scores DataFrame
counties = pivot_table_cleaned.index
df_scores = pd.DataFrame({
    'PC1_Score': scores_matrix[:, 0],
    'PC2_Score': scores_matrix[:, 1],
    'County': counties,
    'Cluster': [f'Group {c+1}' for c in cluster_labels]
})

# 建立 Loadings DataFrame (年份)
years = pivot_table_cleaned.columns
df_loadings = pd.DataFrame({
    'Year': years,
    'PC1_Loading': loadings_matrix[:, 0],
    'PC2_Loading': loadings_matrix[:, 1]
})

# ==========================================
# 4. 視覺化：縣市得分圖 (Scores Plot)
# ==========================================
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df_scores, x='PC1_Score', y='PC2_Score', hue='Cluster', palette='Set1', s=150, alpha=0.8)

# 加上縣市名稱標籤
if 'prop' in locals() or 'prop' in globals():
    font_name = prop.get_name()
else:
    font_name = None

for i, txt in enumerate(df_scores['County']):
    plt.annotate(txt, (df_scores['PC1_Score'].iloc[i], df_scores['PC2_Score'].iloc[i]),
                 textcoords="offset points", xytext=(0,10), ha='center', fontsize=11, fontname=font_name)

plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
plt.title('County Scores Plot (Excluding Lienchiang County)', fontsize=14)
plt.xlabel(f"PC1 Score ({pca_full.explained_variance_ratio_[0]*100:.2f}%)", fontsize=12)
plt.ylabel(f"PC2 Score ({pca_full.explained_variance_ratio_[1]*100:.2f}%)", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(title='Hierarchical Clusters')
plt.tight_layout()
plt.show()

# ==========================================
# 5. 視覺化：年份負荷量隨時間演變圖
# ==========================================
plt.figure(figsize=(12, 5))
plt.plot(df_loadings['Year'], df_loadings['PC1_Loading'], marker='o', label='PC1 Loading', color='navy', linewidth=2)
plt.plot(df_loadings['Year'], df_loadings['PC2_Loading'], marker='s', label='PC2 Loading', color='crimson', linewidth=2)
plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.title('Loadings over Time (Excluding Lienchiang County)', fontsize=14)
plt.xlabel('Diagnosis Year', fontsize=12)
plt.ylabel('Loading Value', fontsize=12)
plt.xticks(df_loadings['Year'][::4], rotation=45)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### PCA 變異解釋量分析視覺化
透過 Scree Plot 與累積變異圖，我們可以評估降維的效果。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 取得解釋變異量比率
exp_var_ratio = pca_full.explained_variance_ratio_
cum_var_ratio = np.cumsum(exp_var_ratio)

fig, ax1 = plt.subplots(figsize=(10, 6))

# 1. 繪製 Scree Plot (左軸)
color = 'tab:blue'
ax1.set_xlabel('Principal Component Index')
ax1.set_ylabel('Explained Variance Ratio', color=color)
ax1.plot(range(1, len(exp_var_ratio) + 1), exp_var_ratio, marker='o', linestyle='--', color=color, label='Individual')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xticks(range(1, len(exp_var_ratio) + 1, 2))
ax1.set_ylim(-.1, 1.1)  # 設定左軸範圍為 (0, 1)

# 2. 繪製 Cumulative Variance Plot (右軸)
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Cumulative Explained Variance', color=color)
ax2.plot(range(1, len(cum_var_ratio) + 1), cum_var_ratio, marker='s', linestyle='-', color=color, label='Cumulative')
ax2.tick_params(axis='y', labelcolor=color)
ax2.set_ylim(-.1, 1.1)  # 設定右軸範圍為 (0, 1)

# 標示 80% 門檻線
ax2.axhline(y=0.8, color='gray', linestyle=':', alpha=0.7)
ax2.text(len(cum_var_ratio)*0.7, 0.82, '80% Threshold', color='gray', fontsize=10)

plt.title('PCA Scree Plot & Cumulative Explained Variance (Excluding Lienchiang)')
fig.tight_layout()
plt.grid(True, alpha=0.2)
plt.show()

### PC1 ~ PC4 負荷量 (Loadings) 分析
我們提取前四個主成分（累積解釋變異量 > 80%），並觀察各年份在這些成分上的負荷量。這有助於定義每個 PC 所代表的「時間特徵」。

In [ ]:
# 1. 提取前四個主成分的 Loadings
pca_4 = PCA(n_components=4)
pca_4.fit(scaled_data)
loadings_4 = pca_4.components_.T * np.sqrt(pca_4.explained_variance_)

# 建立 DataFrame
years = pivot_table_cleaned.columns
df_loadings_4 = pd.DataFrame(loadings_4, index=years, columns=['PC1', 'PC2', 'PC3', 'PC4'])

# 2. 繪製 Loadings 熱圖
plt.figure(figsize=(10, 12))
sns.heatmap(df_loadings_4, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('PCA Loadings Heatmap (PC1 - PC4)', fontsize=14)
plt.ylabel('Year')
plt.show()

In [ ]:
print(loadings_4)

### 縣市得分圖 (Scores Plot): PC3 vs PC4
除了主要的 PC1/PC2，我們觀察 PC3 與 PC4 是否能區分出特定的縣市群體。

In [ ]:
# 計算縣市得分
scores_4 = pca_4.transform(scaled_data)
df_scores_4 = pd.DataFrame(scores_4, index=pivot_table_cleaned.index, columns=['PC1', 'PC2', 'PC3', 'PC4'])

# 繪製 PC3 vs PC4
plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_scores_4, x='PC3', y='PC4', s=100, color='darkorange', alpha=0.7)

# 標記縣市名稱
for i, county in enumerate(df_scores_4.index):
    plt.annotate(county, (df_scores_4['PC3'].iloc[i], df_scores_4['PC4'].iloc[i]),
                 textcoords="offset points", xytext=(0,10), ha='center', fontsize=10, fontname=font_name)

plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
plt.title('County Scores Plot: PC3 vs PC4', fontsize=14)
plt.xlabel(f'PC3 ({pca_4.explained_variance_ratio_[2]*100:.2f}%)')
plt.ylabel(f'PC4 ({pca_4.explained_variance_ratio_[3]*100:.2f}%)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 確保已剔除連江縣，並建立包含分群標籤的 DataFrame
df_plot_raw = pivot_table_age_std.drop(index='連江縣', errors='ignore').copy()

# 假設 df_scores 是我們上一步做好的 DataFrame (內含 County 和 Cluster 欄位)
# 將分群資訊 mapping 回原始的 pivot table
df_plot_raw = df_plot_raw.reset_index()
df_plot_raw = df_plot_raw.merge(df_scores[['County', 'Cluster']], left_on='City_County', right_on='County')

# 2. 將資料由寬轉長 (Melt)，方便 Seaborn 畫折線圖
df_long = df_plot_raw.drop(columns=['County']).melt(
    id_vars=['City_County', 'Cluster'],
    var_name='Year',
    value_name='Incidence_Rate'
)
df_long['Year'] = df_long['Year'].astype(int)

# 3. 開始畫圖：利用 FacetGrid 依群組分開畫，或者全部畫在同一張
plt.figure(figsize=(14, 8))

# 畫出每個縣市自己的線 (estimator=None, units='City_County')
sns.lineplot(
    data=df_long,
    x='Year',
    y='Incidence_Rate',
    hue='Cluster',
    units='City_County',
    estimator=None,
    palette='Set1',
    alpha=0.4,
    linewidth=1.5
)

# 畫出各群組的「平均趨勢粗線」作為對比
sns.lineplot(
    data=df_long,
    x='Year',
    y='Incidence_Rate',
    hue='Cluster',
    palette='Set1',
    linewidth=4,
    legend=False
)

# 特別標記出 2003 年分水嶺
plt.axvline(2003, color='black', linestyle=':', linewidth=2, label='2003 Cancer Control Act')

plt.title('Individual County Lung Cancer Trends Grouped by PCA Clusters', fontsize=16)
plt.xlabel('Diagnosis Year', fontsize=12)
plt.ylabel('Age-Standardized Incidence Rate (per 100k)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()